# Round 1 · Account and profile rules

**The attacker in the data:** a farm. Accounts made this week, messaging dozens of strangers within a day
or two of joining. That is what an operation looks like before anyone gives it a language model.

**Your job:** flag the farm's accounts in `out/suspects_r1.csv` without putting any cohort of real members
over its false-positive budget (2%, at least one person). The checkpoint tells you which cohort you
hurt and why it is a trap, but not who the scammers are.

Starter rule below: "a new account that bursts". Run it, read the scoreboard, then fix it.

In [ ]:
import sys
sys.path.insert(0, "../tools")
import pandas as pd
import trailmates as tm
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

feats = pd.DataFrame.from_dict(tm.member_features(), orient="index")
feats[["age_at_first_outbound_days", "fanout_48h", "photo_reuse", "photo_count_raw", "egress", "default_avatar"]].describe(include="all")

Features you will want:

| feature | meaning |
|---|---|
| `age_at_first_outbound_days` | account age when it first messaged someone it had never talked to |
| `fanout_48h` | most first contacts it sent inside any 48 hours |
| `photo_count_raw` | how many accounts use the same photo, default avatars included |
| `photo_reuse` | the same, but 0 for the community's default avatars |
| `egress` | the kind of network it signed up from: residential, mobile, hosting, vpn |

In [ ]:
def round1(f):
    # TODO (Round 1): the starter flags every new account that bursts. Read who that hurts,
    # then require something the farm cannot hide: reused (uploaded) photos or hosting egress.
    return (f["age_at_first_outbound_days"] <= 7) & (f["fanout_48h"] >= 15)


flagged = feats[round1(feats)]
tm.write_suspects("../out/suspects_r1.csv", {m: 1.0 + r.fanout_48h / 100 for m, r in flagged.iterrows()})
print(len(flagged), "flagged")

In [ ]:
!{sys.executable} ../tools/checkpoint.py --round 1

## Read the scoreboard

- Which cohort is over budget? Look at what those accounts have in common with the farm, and what they don't:

In [ ]:
flagged[["age_at_first_outbound_days", "fanout_48h", "photo_reuse", "photo_count_raw", "egress"]].groupby("egress").describe().T

- Before you add photo reuse: try `photo_count_raw >= 3` on its own and score it. Who shares a photo
  honestly? Then try `photo_reuse`.
- Before you add egress: try hosting/VPN on its own. Who signs up from a VPN?

When `--round 1` prints PASS, go on to `02_shape.ipynb`. Stuck? `python tools/checkpoint.py --round 1 --catch-up`
installs the reference file (yours is kept as `.bak`; `--restore` puts it back), and
`solutions/round1_profile.py` explains its choices.